# Laboratorio de IA ética: Titanic

Modelo base y transformación de variables. Aprendemos las reglas con entrenamiento y evaluamos sobre validación.

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/memo124/Laboratorio_IA_etica/blob/main/main.ipynb)

## 1. Dependencias

Ejecuta las celdas en orden en Colab.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

## 2. Carga y exploración

Cada fila es un pasajero. `Survived` indica supervivencia (1) o fallecimiento (0). Leemos el CSV local o lo descargamos.

In [ ]:
DATA_PATH = Path("train.csv")
DATA_URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

if DATA_PATH.exists():
    data = pd.read_csv(DATA_PATH)
else:
    data = pd.read_csv(DATA_URL)
    # Guardamos una copia para no descargarla otra vez en esta sesión.
    data.to_csv(DATA_PATH, index=False)

Revisamos las primeras filas, los tipos de datos y la cantidad de valores faltantes.

In [ ]:
display(data.head())

data.info()

display(data.isna().sum())

## 3. División de datos

Dividimos en entrenamiento (60 %), validación (20 %) y prueba (20 %). `stratify` conserva la proporción de supervivientes; la semilla 42 permite repetir la división.

In [ ]:
features = data.drop(columns=["Survived"])
target = data["Survived"]

X_remaining, X_test, y_remaining, y_test = train_test_split(
    features,
    target,
    test_size=0.20,
    stratify=target,
    random_state=42,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_remaining,
    y_remaining,
    test_size=0.25,
    stratify=y_remaining,
    random_state=42,
)

print("Pasajeros de entrenamiento:", len(X_train))
print("Pasajeros de validación:", len(X_val))
print("Pasajeros de prueba:", len(X_test))

## 4. Preprocesamiento

Completamos números con la mediana y categorías con el valor más frecuente. One-hot crea columnas de ceros y unos. `Pclass` se trata como categoría.

In [ ]:
numeric_columns = ["Age", "SibSp", "Parch", "Fare"]
categorical_columns = ["Pclass", "Sex", "Embarked"]

numeric_transformer = SimpleImputer(strategy="median")

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_columns),
        ("categorical", categorical_transformer, categorical_columns),
    ],
    verbose_feature_names_out=False,
)

## 5. Modelo base y entrenamiento

El pipeline prepara los datos y entrena un Random Forest de 100 árboles, usando solo entrenamiento.

In [ ]:
classifier = RandomForestClassifier(n_estimators=100, random_state=42)

baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", classifier),
])

baseline_model.fit(X_train, y_train)

## 6. Preparación de validación y prueba

`transform` aplica las reglas aprendidas sin reajustarlas. Prueba queda reservada para la evaluación final.

In [ ]:
fitted_preprocessor = baseline_model.named_steps["preprocessor"]
X_val_prepared = fitted_preprocessor.transform(X_val)
X_test_prepared = fitted_preprocessor.transform(X_test)

# shape muestra la cantidad de filas y columnas resultantes.
print("Validación preparada:", X_val_prepared.shape)
print("Prueba preparada:", X_test_prepared.shape)

## 7. Evaluación del Baseline

Calculamos F1 en validación como referencia. Combina precisión y sensibilidad: cuanto más cerca de 1, mejor.

In [ ]:
y_val_pred = baseline_model.predict(X_val)
validation_f1 = f1_score(y_val, y_val_pred)

print(f"F1 del Baseline en validación: {validation_f1:.4f}")

print(classification_report(y_val, y_val_pred))

## 8. Distribución del precio del boleto

El histograma agrupa los precios en 30 intervalos. Observamos si predominan precios bajos con una cola hacia valores altos.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(X_train["Fare"].dropna(), bins=30, edgecolor="black")
plt.title("Precio del boleto en entrenamiento")
plt.xlabel("Precio del boleto (Fare)")
plt.ylabel("Cantidad de pasajeros")
plt.tight_layout()
plt.show()

## 9. Copias y valores faltantes

Creamos copias y completamos faltantes con las medianas de entrenamiento. Las columnas originales se conservan.

In [ ]:
X_train_engineered = X_train.copy()
X_val_engineered = X_val.copy()
X_test_engineered = X_test.copy()

fare_median = X_train["Fare"].median()
age_median = X_train["Age"].median()

In [ ]:
fare_train_filled = X_train["Fare"].fillna(fare_median)
fare_val_filled = X_val["Fare"].fillna(fare_median)
fare_test_filled = X_test["Fare"].fillna(fare_median)

age_train_filled = X_train["Age"].fillna(age_median)
age_val_filled = X_val["Age"].fillna(age_median)
age_test_filled = X_test["Age"].fillna(age_median)

## 10. Transformación logarítmica

`np.log1p` comprime los precios altos y admite ceros. Guardamos el resultado en `Fare_log` sin reemplazar `Fare`.

In [ ]:
X_train_engineered["Fare_log"] = np.log1p(fare_train_filled)
X_val_engineered["Fare_log"] = np.log1p(fare_val_filled)
X_test_engineered["Fare_log"] = np.log1p(fare_test_filled)

## 11. Histogramas antes y después

Comparamos entrenamiento antes y después del logaritmo. Los ejes horizontales tienen distintas unidades.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

axes[0].hist(fare_train_filled, bins=30, edgecolor="black")
axes[0].set_title("Antes: precio del boleto")
axes[0].set_xlabel("Fare imputado")
axes[0].set_ylabel("Cantidad de pasajeros")

axes[1].hist(X_train_engineered["Fare_log"], bins=30, edgecolor="black")
axes[1].set_title("Después: logaritmo del precio")
axes[1].set_xlabel("Fare_log")
axes[1].set_ylabel("Cantidad de pasajeros")

plt.tight_layout()
plt.show()

El logaritmo comprime la cola de precios altos; esto no garantiza mejores predicciones.

## 12. Revisión de valores extremos

El boxplot muestra la mediana, la dispersión y posibles extremos. Un valor extremo no necesariamente es un error.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].boxplot(X_train["Fare"].dropna())
axes[0].set_title("Precios en entrenamiento")
axes[0].set_xticks([1], ["Fare"])
axes[0].set_ylabel("Precio del boleto")

axes[1].boxplot(X_train["Age"].dropna())
axes[1].set_title("Edades en entrenamiento")
axes[1].set_xticks([1], ["Age"])
axes[1].set_ylabel("Edad (años)")

plt.tight_layout()
plt.show()

## 13. Límites para recortar extremos

Calculamos los percentiles 1 y 99 solo con entrenamiento, ignorando faltantes. Aproximadamente el 1 % queda por debajo del primer límite y el 1 % por encima del segundo.

In [ ]:
fare_lower = X_train["Fare"].quantile(0.01)
fare_upper = X_train["Fare"].quantile(0.99)
age_lower = X_train["Age"].quantile(0.01)
age_upper = X_train["Age"].quantile(0.99)

winsor_limits = pd.DataFrame({
    "Variable": ["Fare", "Age"],
    "Percentil 1": [fare_lower, age_lower],
    "Percentil 99": [fare_upper, age_upper],
})
display(winsor_limits)

## 14. Winsorización del precio y la edad

Winsorizar recorta extremos sin eliminar filas. Aplicamos `clip` con los mismos límites de entrenamiento a las tres particiones.

In [ ]:
X_train_engineered["Fare_winsor"] = fare_train_filled.clip(lower=fare_lower, upper=fare_upper)
X_val_engineered["Fare_winsor"] = fare_val_filled.clip(lower=fare_lower, upper=fare_upper)
X_test_engineered["Fare_winsor"] = fare_test_filled.clip(lower=fare_lower, upper=fare_upper)

In [ ]:
X_train_engineered["Age_winsor"] = age_train_filled.clip(lower=age_lower, upper=age_upper)
X_val_engineered["Age_winsor"] = age_val_filled.clip(lower=age_lower, upper=age_upper)
X_test_engineered["Age_winsor"] = age_test_filled.clip(lower=age_lower, upper=age_upper)

## 15. Boxplots antes y después

Comparamos los valores imputados antes y después del recorte, con la misma escala por variable.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharey="row")

axes[0, 0].boxplot(fare_train_filled)
axes[0, 0].set_title("Antes: precio imputado")
axes[0, 0].set_xticks([1], ["Fare"])
axes[0, 0].set_ylabel("Precio del boleto")

axes[0, 1].boxplot(X_train_engineered["Fare_winsor"])
axes[0, 1].set_title("Después: precio recortado")
axes[0, 1].set_xticks([1], ["Fare_winsor"])

axes[1, 0].boxplot(age_train_filled)
axes[1, 0].set_title("Antes: edad imputada")
axes[1, 0].set_xticks([1], ["Age"])
axes[1, 0].set_ylabel("Edad (años)")

axes[1, 1].boxplot(X_train_engineered["Age_winsor"])
axes[1, 1].set_title("Después: edad recortada")
axes[1, 1].set_xticks([1], ["Age_winsor"])

plt.tight_layout()
plt.show()

El recorte conserva todos los pasajeros. Pueden quedar puntos fuera de los bigotes porque el boxplot usa un criterio distinto de los percentiles.